# 01 — Khám phá dữ liệu Superstore

**Bài toán:** dự báo tổng doanh số (`Sales`) theo tuần để hỗ trợ quản lý bán lẻ. Dữ liệu giao dịch được tổng hợp theo tuần trước khi xây mô hình; chỉ dùng các tuần trọn vẹn để tránh tuần đầu/cuối thiếu ngày.

Notebook này kiểm tra cấu trúc, thiếu dữ liệu, phân phối, ngoại lệ, tương quan và xu hướng thời gian. Quy tắc IQR dùng để đánh dấu quan sát cần xem xét, không tự động xóa giao dịch doanh số lớn vì đó có thể là đơn hàng thật.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

candidates = [Path('dataset/Sample - Superstore.csv'), Path('../dataset/Sample - Superstore.csv')]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Không tìm thấy dataset/Sample - Superstore.csv; chạy notebook từ thư mục smart_learning hoặc notebooks.')
df = pd.read_csv(data_path, encoding='cp1252')
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y', errors='coerce')
df.head()

## 1. Cấu trúc và chất lượng dữ liệu
Kiểm tra kiểu dữ liệu, bản ghi trùng và giá trị thiếu. `Postal Code` là mã địa lý; không xem mã này như đại lượng liên tục trong mô hình.

In [ ]:
print(f'Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột')
print(f'Trùng toàn bộ dòng: {df.duplicated().sum():,}')
display(pd.DataFrame({'kiểu dữ liệu': df.dtypes.astype(str), 'thiếu': df.isna().sum(), 'thiếu (%)': (df.isna().mean() * 100).round(2)}))
display(df.describe(include='all').T)

## 2. Phân phối và ngoại lệ
Các cột tiền thường lệch phải. Biểu đồ log giúp nhìn rõ vùng giao dịch phổ biến; bảng IQR cho biết số điểm nằm ngoài khoảng 1.5 × IQR. Đây là cờ kiểm tra, không phải quy tắc loại dữ liệu.

In [ ]:
numeric_cols = ['Sales', 'Profit', 'Quantity', 'Discount']
summary = df[numeric_cols].describe().T
summary['median'] = df[numeric_cols].median()
summary['skew'] = df[numeric_cols].skew()
summary['missing'] = df[numeric_cols].isna().sum()
outlier_rows = []
for column in numeric_cols:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    count = ((df[column] < q1 - 1.5 * iqr) | (df[column] > q3 + 1.5 * iqr)).sum()
    outlier_rows.append({'cột': column, 'Q1': q1, 'Q3': q3, 'số ngoại lệ IQR': int(count), 'tỷ lệ (%)': count / len(df) * 100})
display(summary.round(3))
display(pd.DataFrame(outlier_rows).set_index('cột').round(2))

fig = px.histogram(df, x='Sales', nbins=60, log_x=True, title='Phân phối Sales (trục x log)', color_discrete_sequence=['#2563EB'])
fig.update_layout(xaxis_title='Sales (log scale)', yaxis_title='Số giao dịch')
fig.show()

## 3. Mối quan hệ giữa các biến số
Tương quan chỉ mô tả quan hệ tuyến tính và không chứng minh nhân quả. `Profit` có thể dùng để khám phá, nhưng không đưa vào mô hình dự báo doanh thu tương lai vì chưa biết giá trị đó tại thời điểm dự báo.

In [ ]:
corr = df[numeric_cols].corr(numeric_only=True)
display(corr.round(3))
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Tương quan Pearson giữa biến số')
fig.show()
fig = px.scatter(df.sample(min(2500, len(df)), random_state=42), x='Discount', y='Profit', color='Category', hover_data=['Sales', 'Quantity'], opacity=.6, title='Discount và Profit theo danh mục')
fig.show()

## 4. Tổng hợp chuỗi doanh số theo tuần
Dữ liệu bao phủ nhiều năm. Tổng doanh số theo tuần là mục tiêu dự báo. Tuần đầu và cuối không đủ 7 ngày nên được loại khỏi chuỗi mô hình để không tạo mức doanh số thấp giả tạo. Các tuần không có giao dịch giữa hai đầu chuỗi được điền 0 sau khi tái lập lịch tuần.

In [ ]:
valid_dates = df['Order Date'].dropna()
first_day, last_day = valid_dates.min(), valid_dates.max()
weekly_all = df.dropna(subset=['Order Date']).set_index('Order Date')['Sales'].resample('W-SUN').sum()
first_week_start = weekly_all.index[0] - pd.Timedelta(days=6)
last_week_start = weekly_all.index[-1] - pd.Timedelta(days=6)
weekly = weekly_all.loc[(weekly_all.index - pd.Timedelta(days=6) >= first_day) & (weekly_all.index <= last_day)].copy()
print(f'Ngày dữ liệu: {first_day.date()} đến {last_day.date()}')
print(f'Số tuần trọn vẹn: {len(weekly):,}; tuần đầu/tuần cuối bị loại vì không trọn 7 ngày')
display(weekly.rename('Sales').describe().to_frame().round(2))
fig = px.line(weekly.rename('Sales').reset_index(names='Week End'), x='Week End', y='Sales', title='Tổng Sales theo tuần trọn vẹn', markers=True)
fig.update_traces(line_color='#2563EB')
fig.show()

weekly_month = df.assign(Month=df['Order Date'].dt.month).groupby('Month')['Sales'].sum().reindex(range(1, 13))
fig = px.bar(x=weekly_month.index, y=weekly_month.values, labels={'x': 'Tháng', 'y': 'Sales'}, title='Sales theo tháng lịch (gộp các năm)', color=weekly_month.values, color_continuous_scale='Blues')
fig.show()

## 5. Kết luận EDA và quyết định tiền xử lý
- Dataset gồm 9.994 giao dịch và 21 cột; ngày đặt hàng trải từ 03/01/2014 đến 30/12/2017.
- Không có giá trị thiếu ở dữ liệu được cung cấp; vẫn cần kiểm tra kiểu ngày và số khi nạp dữ liệu trong ứng dụng.
- `Sales` lệch phải rõ (skew khoảng 12.97); khoảng 1.167 giao dịch vượt ngưỡng IQR. Không xóa các giao dịch đó mặc định.
- Mục tiêu là tổng `Sales` theo tuần trọn vẹn. `Profit`, `Discount`, `Quantity`, địa lý và danh mục hữu ích cho khám phá; chúng không được dùng làm đặc trưng nếu không có sẵn trước thời điểm dự báo hoặc có nguy cơ rò rỉ tương lai.
- Chuỗi tuần có khoảng 207 điểm sau khi bỏ hai tuần biên không đầy đủ. Cần dùng chia tách theo thời gian và expanding-window validation; số điểm hạn chế nên cần báo rõ độ bất định và không dùng xáo trộn ngẫu nhiên.